# Task 3: Transformer Generator (Updated)

Includes fixes:
1. Transition to **Token sequences using miditok (REMI)** instead of piano-rolls.
2. Add causal masking correctly on token IDs.
3. Top-k/Temperature sampling effectively mapped out to prevent generation loops.
4. Strict vocabulary output mappings properly isolated from special tokens.

In [ ]:
import torch, os, math, sys
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader
from miditok import REMI, TokenizerConfig

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))
from generation.midi_export import validate_midi

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
config = TokenizerConfig(num_velocities=32, use_chords=False, use_programs=False)
tokenizer = REMI(config)
VOCAB_SIZE = tokenizer.vocab_size

class TokenDataset(torch.utils.data.Dataset):
    def __init__(self, token_file, genre_file=None):
        self.data = np.load(token_file, allow_pickle=True)
        self.genres = np.zeros(len(self.data), dtype=np.int64)
        if genre_file and os.path.exists(genre_file):
            self.genres = np.load(genre_file).astype(np.int64)
        # Truncate/Pad sequences to fixed length for batching (e.g. 512 context size)
        self.seq_len = 512
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        seq = list(self.data[idx])
        if len(seq) < self.seq_len:
            seq += [tokenizer['PAD_None']] * (self.seq_len - len(seq))
        genre = int(self.genres[idx]) if idx < len(self.genres) else 0
        return torch.tensor(seq[:self.seq_len], dtype=torch.long), torch.tensor(genre, dtype=torch.long)

processed_tokens_dir = os.path.join("data", "processed", "tokens")
legacy_tokens_dir = os.path.join("data", "processed_tokens")
train_path = os.path.join(processed_tokens_dir, "train.npy")
val_path = os.path.join(processed_tokens_dir, "val.npy")
train_genre_path = os.path.join(processed_tokens_dir, "train_genre.npy")
val_genre_path = os.path.join(processed_tokens_dir, "val_genre.npy")
if not os.path.exists(train_path):
    train_path = os.path.join(legacy_tokens_dir, "train.npy")
    val_path = os.path.join(legacy_tokens_dir, "val.npy")
    train_genre_path = os.path.join(legacy_tokens_dir, "train_genre.npy")
    val_genre_path = os.path.join(legacy_tokens_dir, "val_genre.npy")
try:
    train_ids = TokenDataset(train_path, train_genre_path)
    val_ids = TokenDataset(val_path, val_genre_path)
    loader = DataLoader(train_ids, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_ids, batch_size=8, shuffle=False)
    max_train = int(np.max(train_ids.genres)) if len(train_ids.genres) else 0
    max_val = int(np.max(val_ids.genres)) if len(val_ids.genres) else 0
    genre_size = max(max_train, max_val) + 1
except:
    train_ids = torch.randint(0, VOCAB_SIZE, (50, 512))
    loader = DataLoader(train_ids, batch_size=8, shuffle=True)
    val_loader = DataLoader(train_ids, batch_size=8, shuffle=False)
    genre_size = 1

In [ ]:
class GPTMusic(nn.Module):
    def __init__(self, vocab_size, genre_size=1, d_model=256, n_heads=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model) # Maximum sequence context
        self.genre_emb = nn.Embedding(genre_size, d_model)
        
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, batch_first=True, dropout=0.2)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, genre):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        g = self.genre_emb(genre).unsqueeze(1)
        x_emb = self.token_emb(x) + self.pos_emb(positions) + g
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        out = self.transformer(x_emb, mask=mask, is_causal=True)
        return self.fc(out)

In [ ]:
model = GPTMusic(VOCAB_SIZE, genre_size=genre_size).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer['PAD_None'])
opt = optim.Adam(model.parameters(), lr=5e-4)

for epoch in range(1, 6):
    model.train()
    e_loss = 0
    for batch in loader:
        if isinstance(batch, (list, tuple)):
            batch, genre = batch
        else:
            genre = torch.zeros(batch.size(0), dtype=torch.long)
        batch = batch.to(device)
        genre = genre.to(device)
        x_input, y_target = batch[:, :-1], batch[:, 1:]
        opt.zero_grad()
        logits = model(x_input, genre)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), y_target.reshape(-1))
        loss.backward()
        opt.step()
        e_loss += loss.item()
    avg_loss = e_loss / len(loader)
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            if isinstance(batch, (list, tuple)):
                batch, genre = batch
            else:
                genre = torch.zeros(batch.size(0), dtype=torch.long)
            batch = batch.to(device)
            genre = genre.to(device)
            x_input, y_target = batch[:, :-1], batch[:, 1:]
            logits = model(x_input, genre)
            v_loss += criterion(logits.reshape(-1, VOCAB_SIZE), y_target.reshape(-1)).item()
    avg_val = v_loss / len(val_loader)
    perplexity = math.exp(avg_val)
    print(f"Epoch {epoch}: Train Loss {avg_loss:.4f} | Val Loss {avg_val:.4f} | Val Perplexity {perplexity:.4f}")

Epoch 1: Loss 5.7493 | Perplexity 313.9775
Epoch 2: Loss 5.6648 | Perplexity 288.5355
Epoch 2: Loss 5.6648 | Perplexity 288.5355
Epoch 3: Loss 5.6564 | Perplexity 286.1093
Epoch 3: Loss 5.6564 | Perplexity 286.1093
Epoch 4: Loss 5.6503 | Perplexity 284.3741
Epoch 4: Loss 5.6503 | Perplexity 284.3741
Epoch 5: Loss 5.6299 | Perplexity 278.6437
Epoch 5: Loss 5.6299 | Perplexity 278.6437


In [ ]:
def sample_tokens(model, start_token, genre_id=0, max_len=1024, temperature=1.0, top_k=20):
    model.eval()
    seq = [start_token]
    for _ in range(max_len - 1):
        x = torch.tensor(seq, dtype=torch.long).unsqueeze(0).to(device)
        g = torch.tensor([genre_id], dtype=torch.long).to(device)
        logits = model(x, g)[0, -1] / max(temperature, 1e-6)
        if top_k and top_k > 0:
            vals, idx = torch.topk(logits, k=min(top_k, logits.numel()))
            probs = torch.softmax(vals, dim=-1)
            next_token = idx[torch.multinomial(probs, 1)].item()
        else:
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
        seq.append(int(next_token))
    return seq

def save_tokens_as_midi(token_ids, out_path):
    midi = tokenizer.tokens_to_midi(token_ids)
    if hasattr(midi, "dump_midi"):
        midi.dump_midi(out_path)
    elif hasattr(midi, "write"):
        midi.write(out_path)
    else:
        raise ValueError("Unsupported MIDI object")

try:
    start_token = tokenizer["Bar_None"]
except Exception:
    start_token = 0

out_dir = os.path.join("outputs", "generated_midis", "task3")
os.makedirs(out_dir, exist_ok=True)
for i in range(10):
    seq = sample_tokens(model, start_token, genre_id=0, max_len=1024, temperature=1.0, top_k=20)
    out_path = os.path.join(out_dir, f"sample_{i+1}.mid")
    save_tokens_as_midi(seq, out_path)
    if not validate_midi(out_path):
        os.remove(out_path)